# 🚀 MetaGen Brain: Llama 3.1 8B Instruct Fine-Tuning Pipeline

This notebook provides the complete, end-to-end pipeline to fine-tune **Meta Llama 3.1 8B Instruct** on YouTube SEO metadata datasets using **QLoRA (4-bit)** with Llama 3 special tokens, merge the trained adapter weights with ultra-low RAM streaming, compile the model into a lightweight **Q4_K_M GGUF format (~4.9 GB)** via `llama.cpp`, and deploy it to Hugging Face Hub for lightning-fast CPU/GPU local execution in MetaGen.

In [ ]:
# 1. Install required packages for Llama 3.1 QLoRA fine-tuning & GGUF conversion
!pip install -q -U "transformers>=4.49.0" "datasets>=2.19.0" "accelerate>=0.32.0" "peft>=0.11.1" "bitsandbytes>=0.43.1" "huggingface_hub>=0.23.0" safetensors

In [ ]:
# 2. Check GPU Environment
import torch

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("Warning: GPU not detected. Make sure you are using a GPU runtime in Colab.")

In [ ]:
# 3. Hugging Face Login (Make sure HF_TOKEN is configured in Colab Secrets)
from huggingface_hub import login
import os

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print("Successfully logged in to Hugging Face Hub!")
except Exception as e:
    print("Colab secret HF_TOKEN not found, please authenticate manually:")
    login()

## ⚙️ Model & Tokenizer Configuration (Meta Llama 3.1 8B Instruct with SDPA)

In [ ]:
# 4. Load Llama 3.1 8B with 4-bit Quantization & Fast SDPA Kernels
import torch
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Clear residual VRAM from previous runs
for var in ['model', 'trainer', 'base_model', 'merged_model']:
    if var in globals():
        del globals()[var]
gc.collect()
torch.cuda.empty_cache()

MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map={"": 0},
    dtype=torch.bfloat16,
    attn_implementation="sdpa",
)

print(f"Loaded {MODEL_ID} in 4-bit with SDPA acceleration successfully!")

## 📝 Dataset Loading (Google Drive / Local), Formatting & Tokenization

In [ ]:
# 5. Mount Google Drive, Load, Format & Tokenize Dataset
import os
from datasets import load_dataset

DRIVE_PATH = "/content/drive/MyDrive/Colab Notebooks/youtube_finetuning_dataset.json"
LOCAL_PATH = "youtube_finetuning_dataset.json"

if not os.path.exists(DRIVE_PATH) and not os.path.exists(LOCAL_PATH):
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        print(f"Drive mount notice: {e}")

DATASET_PATH = DRIVE_PATH if os.path.exists(DRIVE_PATH) else LOCAL_PATH
dataset = load_dataset("json", data_files=DATASET_PATH, split="train")
print(f"Successfully loaded {len(dataset)} training examples from: {DATASET_PATH}")

def format_llama3_prompt(sample):
    system_msg = "You are a professional YouTube SEO specialist. Output ONLY valid JSON containing title, description, and tags."
    user_msg = f"Generate YouTube SEO metadata for this video script:\n{sample.get('script', sample.get('input', ''))}"
    assistant_msg = sample.get('target_json', sample.get('output', ''))
    
    formatted_text = (
        f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n"
        f"{system_msg}<|eot_id|>"
        f"<|start_header_id|>user<|end_header_id|>\n\n"
        f"{user_msg}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>\n\n"
        f"{assistant_msg}<|eot_id|>"
    )
    return {"text": formatted_text}

dataset = dataset.map(format_llama3_prompt)

def tokenize_fn(examples):
    return tokenizer(examples["text"], truncation=True, max_length=384)

tokenized_dataset = dataset.map(tokenize_fn, batched=True, remove_columns=dataset.column_names)
print("Sample Formatted Llama 3 Prompt:")
print(dataset[0]["text"][:400])
print(f"✅ Tokenized {len(tokenized_dataset)} samples successfully!")

## 🎯 LoRA Configuration for Llama 3 Architecture

In [ ]:
# 6. Configure LoRA Parameters
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

## 🚀 Lightning Fast Training Execution (~5-6 Mins on T4)

In [ ]:
# 7. Setup Training Arguments & Native Trainer (Optimized for ~5-6 mins)
import os
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir="./results_llama3",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    warmup_steps=10,
    max_steps=60,
    max_grad_norm=0.3,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    logging_steps=10,
    optim="paged_adamw_8bit",
    bf16=True,
    fp16=False,
    save_strategy="no",
    report_to="none",
)

trainer = Trainer(
    model=model,
    train_dataset=tokenized_dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
    args=training_args,
)

# Ensure trainable LoRA weights are Float32 for stable T4 training
for param in trainer.model.parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)

print("🚀 Starting Lightning LoRA training (60 steps total)...\n")
trainer.train()

# Save locally
trainer.model.save_pretrained("./final_llama3_adapter")
tokenizer.save_pretrained("./final_llama3_adapter")

# Auto-save permanent copy to Google Drive
drive_adapter_dir = "/content/drive/MyDrive/Colab Notebooks/final_llama3_adapter"
try:
    trainer.model.save_pretrained(drive_adapter_dir)
    tokenizer.save_pretrained(drive_adapter_dir)
    print(f"✅ Adapter permanently backed up to Google Drive: {drive_adapter_dir}")
except Exception as e:
    print(f"Drive backup notice: {e}")

print("✅ Training complete in ~5 mins! Adapter saved successfully.")

## 🔄 Ultra-Low-RAM Safetensors Streaming Merger (< 2 GB Peak RAM)

In [ ]:
# 8. Ultra-Low-RAM Safetensors Direct Streaming Merger (< 2 GB RAM!)
import os
import json
import torch
import gc
import shutil
from huggingface_hub import snapshot_download
from safetensors.torch import load_file, save_file
from transformers import AutoTokenizer

MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
output_dir = "./llama3_yt_seo_merged"
adapter_path = "./final_llama3_adapter"
if not os.path.exists(adapter_path):
    adapter_path = "/content/drive/MyDrive/Colab Notebooks/final_llama3_adapter"

print(f"1. Loading LoRA adapter weights from: {adapter_path}...")
with open(os.path.join(adapter_path, "adapter_config.json"), "r") as f:
    adapter_config = json.load(f)
r = adapter_config.get("r", 16)
lora_alpha = adapter_config.get("lora_alpha", 32)
scaling = lora_alpha / r

lora_weights = load_file(os.path.join(adapter_path, "adapter_model.safetensors"))
print(f"Loaded {len(lora_weights)} LoRA tensors (Scaling factor: {scaling:.2f})")

print(f"2. Locating base model files for {MODEL_ID}...")
base_model_dir = snapshot_download(repo_id=MODEL_ID, allow_patterns=["*.json", "*.safetensors", "tokenizer*"])

os.makedirs(output_dir, exist_ok=True)
for f in os.listdir(base_model_dir):
    if f.endswith(".json") or f.startswith("tokenizer") or f.endswith(".txt"):
        shutil.copy(os.path.join(base_model_dir, f), os.path.join(output_dir, f))

safetensor_files = [f for f in os.listdir(base_model_dir) if f.endswith(".safetensors")]
print(f"3. Merging {len(safetensor_files)} shards layer-by-layer (< 2 GB peak RAM)...")

for shard_file in sorted(safetensor_files):
    print(f"  -> Processing shard {shard_file}...")
    shard_weights = load_file(os.path.join(base_model_dir, shard_file))
    merged_shard = {}
    
    for key, tensor in shard_weights.items():
        lora_a_key = f"base_model.model.{key}".replace(".weight", ".lora_A.weight")
        lora_b_key = f"base_model.model.{key}".replace(".weight", ".lora_B.weight")
        
        if lora_a_key in lora_weights and lora_b_key in lora_weights:
            w_a = lora_weights[lora_a_key].to(torch.float32)
            w_b = lora_weights[lora_b_key].to(torch.float32)
            delta = (w_b @ w_a) * scaling
            merged_tensor = tensor.to(torch.float32) + delta
            merged_shard[key] = merged_tensor.to(tensor.dtype)
        else:
            merged_shard[key] = tensor
            
    save_file(merged_shard, os.path.join(output_dir, shard_file), metadata={"format": "pt"})
    del shard_weights
    del merged_shard
    gc.collect()

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.save_pretrained(output_dir)
print(f"✅ Ultra-Low-RAM merge completed successfully! Model saved to {output_dir}")

## 💾 Backup Merged Model to Google Drive & Download ZIP

In [ ]:
# 8.5 Save Merged Model to Google Drive & Download ZIP to Local Machine
import os
import shutil
from google.colab import files, drive

# Ensure Google Drive is mounted
if not os.path.exists("/content/drive"):
    try:
        drive.mount("/content/drive")
    except Exception as e:
        print(f"Drive mount notice: {e}")

# 1. Copy full merged folder to Google Drive
drive_backup_dir = "/content/drive/MyDrive/Colab Notebooks/llama3_yt_seo_merged"
print(f"1. Backing up merged model to Google Drive at: {drive_backup_dir}...")
try:
    if os.path.exists(drive_backup_dir):
        shutil.rmtree(drive_backup_dir)
    shutil.copytree("./llama3_yt_seo_merged", drive_backup_dir)
    print("✅ Merged model folder successfully backed up to Google Drive!")
except Exception as e:
    print(f"Drive copy notice: {e}")

# 2. Compress merged model into ZIP archive
zip_filename = "llama3_yt_seo_merged.zip"
print(f"2. Compressing model into {zip_filename}...")
!zip -r llama3_yt_seo_merged.zip ./llama3_yt_seo_merged

# 3. Copy ZIP to Google Drive
try:
    shutil.copy(zip_filename, f"/content/drive/MyDrive/Colab Notebooks/{zip_filename}")
    print(f"✅ {zip_filename} also backed up to Google Drive!")
except Exception as e:
    print(f"ZIP Drive copy notice: {e}")

# 4. Trigger direct browser download to local machine
print(f"3. Triggering direct browser download of {zip_filename} to your machine...")
try:
    files.download(zip_filename)
except Exception as e:
    print(f"Browser download notice: {e}")
print("✅ Merged model backup complete!")

## 📦 Quantize to GGUF (llama.cpp) & Push to Hugging Face Hub

In [ ]:
# 9. Compile llama.cpp and convert to Q4_K_M GGUF
import os
from huggingface_hub import HfApi

print("1. Building llama.cpp engine...")
if not os.path.exists("llama.cpp"):
    !git clone https://github.com/ggerganov/llama.cpp
    !cd llama.cpp && cmake -B build && cmake --build build --config Release -j 2
    !pip install -r llama.cpp/requirements.txt

print("2. Converting model to GGUF...")
!python llama.cpp/convert_hf_to_gguf.py ./llama3_yt_seo_merged --outtype f16 --outfile custom-llama3-yt-seo-f16.gguf

print("3. Quantizing to Q4_K_M...")
!./llama.cpp/build/bin/llama-quantize custom-llama3-yt-seo-f16.gguf custom-llama3-yt-seo-Q4_K_M.gguf Q4_K_M || ./llama.cpp/build/llama-quantize custom-llama3-yt-seo-f16.gguf custom-llama3-yt-seo-Q4_K_M.gguf Q4_K_M

print("4. Uploading to Hugging Face Hub...")
api = HfApi()
api.upload_file(
    path_or_fileobj="custom-llama3-yt-seo-Q4_K_M.gguf",
    path_in_repo="custom-llama3-yt-seo-Q4_K_M.gguf",
    repo_id="SujalChhajed925/yt-seo-llama3-merged",
    repo_type="model",
)
print("✅ custom-llama3-yt-seo-Q4_K_M.gguf uploaded to Hugging Face Hub successfully!")